# 02. JSON to Markdown Parsing & Section Routing

This notebook demonstrates how tagged raw JSON blocks are mapped into formatted Markdown documents and organized into section-specific directory hierarchies (`src/step2_parser.py`).

### Key Demonstration Steps:
1. **Slug & Section Extraction**: Regex parsing of chapter titles into standardized directory names and filenames.
2. **Markdown Syntax Mapping**: Converting structural tags (`HEADER`, `SHLOKA`, `BODY`, `FOOTNOTE`) into valid Markdown syntax.
3. **Document Layout Verification**: Rendering a sample chapter to verify heading hierarchy, blockquotes, and footnote sections.

In [1]:
import sys
import json
from pathlib import Path

# Add project root to path for modular imports
sys.path.append("..")

from src.step2_parser import extract_section_and_slug
from src.config import PROCESSED_DATA_DIR

## Step 1: Section Folder & Filename Slug Extraction

Test `extract_section_and_slug` against various Vachanamrut section titles to verify clean directory routing and file naming.

In [2]:
sample_titles = [
    "Gadhada I-1: Continuously Engaging One's Mind on God",
    "Sarangpur-7: Naimisharanya Kshetra",
    "Vartal-11: The Destruction of the Jiva",
    "Additional: Bhugol-Khagol (Geography and Astronomy)"
]

print("--- Section & Slug Parsing ---")
for title in sample_titles:
    folder, slug = extract_section_and_slug(title)
    print(f"Input Title : {title}")
    print(f"Target Dir  : {folder}/")
    print(f"Filename    : {slug}.md\n")

--- Section & Slug Parsing ---
Input Title : Gadhada I-1: Continuously Engaging One's Mind on God
Target Dir  : Gadhada_I/
Filename    : Gadhada_I_1.md

Input Title : Sarangpur-7: Naimisharanya Kshetra
Target Dir  : Sarangpur/
Filename    : Sarangpur_7.md

Input Title : Vartal-11: The Destruction of the Jiva
Target Dir  : Vartal/
Filename    : Vartal_11.md

Input Title : Additional: Bhugol-Khagol (Geography and Astronomy)
Target Dir  : Additional/
Filename    : Additional.md



## Step 2: Render Raw JSON Blocks to Markdown Format

Transform a sample chapter's tagged blocks (`HEADER`, `SHLOKA`, `BODY`, `FOOTNOTE`) into clean, readable Markdown syntax.

In [3]:
# Sample chapter data structure matching step1 scraper output
sample_chapter = {
    "chapter_id": 1,
    "title": "Gadhada I-1: Continuously Engaging One's Mind on God",
    "blocks": [
        "HEADER::Gadhada I-1: Continuously Engaging One's Mind on God",
        "BODY::On the night of Tuesday, Magshar sudi 4, Samvat 1876, Shriji Maharaj was seated in Gadhada.",
        "SHLOKA::Nijatmanam brahmarupam dehatraya-vilakshanam...",
        "BODY::He then asked a question: 'By what means does the mind remain constantly engaged on God?'",
        "FOOTNOTE::1. The three bodies: Sthul, Sukshma, and Karan."
    ]
}

# Apply block transformation logic from step2_parser.py
md_lines = []
has_footnotes = False

for block in sample_chapter["blocks"]:
    prefix, content = block.split("::", 1)
    if prefix == "HEADER":
        md_lines.append(f"# {content}\n")
    elif prefix == "SHLOKA":
        md_lines.append(f"> *{content}*\n")
    elif prefix == "BODY":
        md_lines.append(f"{content}\n")
    elif prefix == "FOOTNOTE":
        if not has_footnotes:
            md_lines.append("## Footnotes\n")
            has_footnotes = True
        md_lines.append(f"* {content}")

rendered_md = "\n".join(md_lines)

print("--- Rendered Markdown Output ---")
print(rendered_md)

--- Rendered Markdown Output ---
# Gadhada I-1: Continuously Engaging One's Mind on God

On the night of Tuesday, Magshar sudi 4, Samvat 1876, Shriji Maharaj was seated in Gadhada.

> *Nijatmanam brahmarupam dehatraya-vilakshanam...*

He then asked a question: 'By what means does the mind remain constantly engaged on God?'

## Footnotes

* 1. The three bodies: Sthul, Sukshma, and Karan.


## Step 3: Target File Path Verification

Verify the target output path for saved Markdown files within `PROCESSED_DATA_DIR`.

In [4]:
folder, slug = extract_section_and_slug(sample_chapter["title"])
target_file_path = PROCESSED_DATA_DIR / "vachanamruts" / folder / f"{slug}.md"

print(f"Target Output Path: {target_file_path}")

Target Output Path: C:\Users\Lenovo\projects\Active Vachanamrut RAG project\data\processed\vachanamruts\Gadhada_I\Gadhada_I_1.md
